# 01_transcribe.ipynb — Audio EDA, Verbatim Transcription & Fluency Feature Extraction

Welcome to the **SHL Kaggle Hiring Assessment 2026** transcription pipeline!

### Task Overview
- **Goal**: Predict continuous grammar score (0–5, MOS Likert scale) from 45–60 s English speech audio (`.wav`).
- **Input Data**:
  - `DATA_DIR/train/*.wav` (769 files)
  - `DATA_DIR/test/*.wav` (216 files)
  - `DATA_DIR/train.csv`, `DATA_DIR/test.csv`, `DATA_DIR/sample_submission.csv`
- **Output of this Notebook**: `/kaggle/working/transcripts.csv` containing verbatim text transcripts, speech rate, pause statistics, and Whisper confidence metrics.

---

### Key Data Handling Rules
1. **Split-aware paths**: Train and test folders use identical filenames (e.g., `audio_0.wav` exists in both train and test). We always track `split` ('train' vs 'test') and build file paths as `DATA_DIR/<split>/<filename>`.
2. **CSV-driven file listing**: Audio IDs are non-contiguous. Filenames must always be read from the CSVs.
3. **Filename Normalization**: Filenames in CSVs may or may not include the `.wav` extension. We normalize this when building audio file paths, while preserving the exact CSV string for submissions.
4. **Placeholder Test Labels**: `test.csv` contains random placeholder numbers — we ignore test labels completely.

## 1. Configuration & Global Parameters

In this cell, we define paths and execution flags:
- `DATA_DIR`: Path to the competition dataset on Kaggle.
- `OUTPUT_DIR`: Kaggle working directory where output CSV files are saved.
- `QUICK_TEST`: Set to `True` to process only the first 5 train and 5 test files for quick testing. Set to `False` for full execution (769 train + 216 test files).

In [ ]:
import os

# Define input data directory and output path on Kaggle
DATA_DIR = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
OUTPUT_DIR = "/kaggle/working"

# Control flag: True = process first 5 train + 5 test audio files for quick pipeline verification.
# Set QUICK_TEST = False when ready for the full Kaggle run!
QUICK_TEST = True

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"[CONFIG] DATA_DIR: {DATA_DIR}")
print(f"[CONFIG] OUTPUT_DIR: {OUTPUT_DIR}")
print(f"[CONFIG] QUICK_TEST mode: {QUICK_TEST}")

## 2. Setup & GPU Diagnostic

We install `faster-whisper` quietly for fast transcription using CTranslate2.
> **Note**: We only install `faster-whisper`. We do **not** re-install PyTorch or CUDA libraries on Kaggle because Kaggle environments come pre-configured with GPU-accelerated PyTorch. Re-installing torch can break GPU support.

In [ ]:
# Install faster-whisper quietly without modifying Kaggle's system PyTorch installation
!pip install -q faster-whisper

import torch

# Check GPU availability and print device diagnostic information
print("--- GPU Diagnostic Info ---")
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

print("\n--- PyTorch CUDA Status ---")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")

### GPU / cuDNN Library Error Fallback (Troubleshooting Note)

If `faster-whisper` fails to load on the GPU due to a cuDNN version or library mismatch (e.g. `libcudnn_ops.so.9 not found` or CUDNN runtime error), you can use the commented code below to set environment paths or adjust compute settings.

In [ ]:
# ==============================================================================
# FALLBACK CELL (Only run if faster-whisper throws cuDNN or CUDA library errors)
# ==============================================================================
#
# Option A: Add system CUDA library paths if libcudnn is not found in standard PATH
# import os
# os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":/usr/local/cuda/lib64"
#
# Option B: Change Whisper compute_type to float32 or int8_float16 if float16 fails
# WHISPER_COMPUTE_TYPE = "float16"  # Try "float32" or "int8_float16" as fallbacks
#
# Option C: Fallback to CPU execution if CUDA environment is critically broken
# WHISPER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Exploratory Data Analysis (EDA)

In this section:
1. Load all three CSV files (`train.csv`, `test.csv`, `sample_submission.csv`) and dynamically inspect column names.
2. Analyze target grammar scores (distribution, MOS Likert statistics).
3. Read audio metadata (duration & sample rate) using fast header inspection via `soundfile.info()`.
4. Verify file integrity (ensure every CSV filename exists on disk and identify any unlisted files).

In [ ]:
import os
import pandas as pd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Set random seed for reproducibility
np.random.seed(42)

# Load CSV files
train_csv_path = os.path.join(DATA_DIR, "train.csv")
test_csv_path = os.path.join(DATA_DIR, "test.csv")
sub_csv_path = os.path.join(DATA_DIR, "sample_submission.csv")

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)
sub_df = pd.read_csv(sub_csv_path)

print(f"train.csv shape: {train_df.shape}")
print(f"test.csv shape:  {test_df.shape}")
print(f"sample_sub shape: {sub_df.shape}\n")

print("train.csv columns:", list(train_df.columns))
print("test.csv columns: ", list(test_df.columns))
print("sample_sub columns:", list(sub_df.columns))

# Dynamic column detection (never assume hardcoded names)
filename_col = None
label_col = None

for col in train_df.columns:
    col_lower = col.lower()
    if any(k in col_lower for k in ["file", "wav", "id", "name", "audio"]):
        filename_col = col
    elif any(k in col_lower for k in ["score", "label", "grammar", "target", "mos"]):
        label_col = col

# Fallbacks if keyword matching didn't catch specific names
if filename_col is None:
    filename_col = train_df.columns[0]
if label_col is None:
    label_col = train_df.columns[1] if len(train_df.columns) > 1 else train_df.columns[0]

print(f"\n[DETECTED COLUMNS] Filename Column: '{filename_col}' | Label Column: '{label_col}'")

print("\ntrain_df.head():")
display(train_df.head())

print("\ntest_df.head():")
display(test_df.head())

In [ ]:
# Analyze Training Label Distribution
labels = train_df[label_col]

print("--- Grammar Target Label Summary Statistics ---")
print(f"Count:  {labels.count()}")
print(f"Mean:   {labels.mean():.4f}")
print(f"Std:    {labels.std():.4f}")
print(f"Min:    {labels.min():.4f}")
print(f"Median: {labels.median():.4f}")
print(f"Max:    {labels.max():.4f}")
print("\nValue Counts (Top 10):")
print(labels.value_counts().head(10))

# Plot Label Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(labels, kde=True, bins=20, ax=axes[0], color='skyblue')
axes[0].set_title("Training Grammar Scores Histogram")
axes[0].set_xlabel("Grammar Score (MOS Likert 0–5)")
axes[0].set_ylabel("Count")

sns.boxplot(x=labels, ax=axes[1], color='lightgreen')
axes[1].set_title("Training Grammar Scores Boxplot")
axes[1].set_xlabel("Grammar Score")

plt.tight_layout()
plt.show()

In [ ]:
# Helper function to construct full file path normalized with .wav extension
def build_audio_path(split_name, filename_str):
    fname = str(filename_str).strip()
    if not fname.lower().endswith(".wav"):
        fname = fname + ".wav"
    return os.path.join(DATA_DIR, split_name, fname)

# Read duration and sample rate for all audio files using fast header inspection (soundfile.info)
def extract_audio_metadata(df, split_name):
    records = []
    for idx, row in df.iterrows():
        orig_fname = row[filename_col]
        full_path = build_audio_path(split_name, orig_fname)
        
        try:
            info = sf.info(full_path)
            records.append({
                'split': split_name,
                'filename': orig_fname,
                'duration_sec': info.duration,
                'samplerate': info.samplerate,
                'channels': info.channels,
                'file_exists': True
            })
        except Exception as e:
            records.append({
                'split': split_name,
                'filename': orig_fname,
                'duration_sec': np.nan,
                'samplerate': np.nan,
                'channels': np.nan,
                'file_exists': False
            })
    return pd.DataFrame(records)

print("[INFO] Inspecting audio file headers for train and test sets...")
train_audio_meta = extract_audio_metadata(train_df, "train")
test_audio_meta = extract_audio_metadata(test_df, "test")

audio_meta_df = pd.concat([train_audio_meta, test_audio_meta], ignore_index=True)

print(f"\nTotal audio files inspected: {len(audio_meta_df)}")
print(f"Missing audio files count:   {sum(~audio_meta_df['file_exists'])}\n")

# Summary of duration & sample rates
for split in ['train', 'test']:
    sub = audio_meta_df[audio_meta_df['split'] == split]
    print(f"--- {split.upper()} Audio Stats ---")
    print(f"Duration Min:  {sub['duration_sec'].min():.2f} s")
    print(f"Duration Mean: {sub['duration_sec'].mean():.2f} s")
    print(f"Duration Max:  {sub['duration_sec'].max():.2f} s")
    print(f"Unique Sample Rates: {sub['samplerate'].unique()}")

# Plot Overlaid Duration Histogram
plt.figure(figsize=(10, 5))
sns.histplot(data=audio_meta_df, x='duration_sec', hue='split', element='step', stat='density', common_norm=False, bins=25)
plt.title("Audio Duration Distribution (Train vs Test)")
plt.xlabel("Duration (seconds)")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Check disk-vs-CSV file alignment
print("--- File Integrity & Disk Alignment Verification ---")

def verify_disk_files(df, split_name):
    missing_in_csv = []
    disk_dir = os.path.join(DATA_DIR, split_name)
    
    # Check CSV -> Disk
    csv_filenames = set()
    for fname in df[filename_col]:
        fname_norm = str(fname).strip()
        if not fname_norm.lower().endswith(".wav"):
            fname_norm += ".wav"
        csv_filenames.add(fname_norm)
        
        path = os.path.join(disk_dir, fname_norm)
        if not os.path.exists(path):
            missing_in_csv.append(fname)
            
    # Check Disk -> CSV
    disk_files = set(f for f in os.listdir(disk_dir) if f.lower().endswith(".wav")) if os.path.exists(disk_dir) else set()
    unlisted_on_disk = disk_files - csv_filenames
    
    print(f"[{split_name.upper()}] Files in CSV missing from disk: {len(missing_in_csv)}")
    if missing_in_csv:
        print(f"  Missing files sample: {missing_in_csv[:5]}")
        
    print(f"[{split_name.upper()}] Audio files on disk NOT listed in CSV: {len(unlisted_on_disk)}")
    if unlisted_on_disk:
        print(f"  Unlisted disk files sample: {list(unlisted_on_disk)[:5]}")

verify_disk_files(train_df, "train")
verify_disk_files(test_df, "test")

## 4. Transcription & Feature Extraction Strategy (`faster-whisper`)

To assess continuous grammar scores accurately, standard ASR transcription is **not enough**. Default Whisper models are trained to produce clean, grammatically polished text — they silently correct grammatical errors, drop hesitation sounds ("uh", "um"), and smooth out stutters.

For grammar scoring, we need a **VERBATIM** transcription that retains all natural speech errors and disfluencies.

### PyAV Bypass Strategy
> **Why we load audio ourselves**: `faster-whisper` internally uses PyAV (`av.open()`) to decode audio files from disk. In some Kaggle environments, version mismatches between PyAV and system ffmpeg raise errors like `open() got an unexpected keyword argument 'metadata_errors'`. To completely bypass PyAV, we pre-load audio files into normalized 1D 16kHz float32 NumPy arrays using `soundfile` (with `librosa.resample` fallback) and pass the raw array directly to `model.transcribe(audio, ...)`.

### Parameter Tuning Rationale
1. `model = WhisperModel("large-v3", device="cuda", compute_type="float16")`: `large-v3` provides highest accuracy word boundary detection and acoustic representation.
2. `language="en"`: Prevents unnecessary language detection overhead and avoids accidental language switching.
3. `beam_size=5`: Uses beam search to find optimal acoustic path matches.
4. `temperature=0.0`: Greedy, deterministic decoding that eliminates hallucination noise.
5. `condition_on_previous_text=False`: Prevents Whisper from conditioning on previous audio chunks, which stops transcript repetition loops and error compounding.
6. `vad_filter=False`: Disables Voice Activity Detection filtering to prevent clipping soft fillers, unvoiced speech, or stutters.
7. `word_timestamps=True`: Extracts precise start and end timestamps for every word to measure speech pauses and word cadence.
8. `initial_prompt`: Warm-starts Whisper's context decoder with example fillers and non-standard grammar (`"Umm, let me think like, hmm... Okay, here's what I'm, like, thinking. I goes to the, uh, the store yesterday."`). This prompts Whisper to keep human speech errors instead of fixing them.

> **Implementation Note**: `faster-whisper` returns a lazy generator for segments. We must convert `segments = list(segments)` before iterating.

In [ ]:
import time
import soundfile as sf
import librosa
from faster_whisper import WhisperModel

# Initialize faster-whisper large-v3 model
print("[INFO] Loading faster-whisper large-v3 model on CUDA...")
model_load_start = time.time()

device_str = "cuda" if torch.cuda.is_available() else "cpu"
model = WhisperModel("large-v3", device=device_str, compute_type="float16")

print(f"[INFO] Model loaded successfully in {time.time() - model_load_start:.2f} seconds.")

# Verbatim prompt to encourage retaining disfluencies and filler words
VERBATIM_INITIAL_PROMPT = (
    "Umm, let me think like, hmm... Okay, here's what I'm, like, thinking. "
    "I goes to the, uh, the store yesterday."
)

def process_audio_file(model, audio_path):
    """
    Loads audio via soundfile to bypass PyAV, ensures 16kHz mono float32 NumPy array,
    transcribes audio verbatim with faster-whisper, and extracts speech rate & pause metrics.
    """
    # 1. Read audio via soundfile to bypass PyAV
    audio, sr = sf.read(audio_path, dtype="float32")
    
    # 2. If stereo (2-D), average channels to mono
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
        
    # 3. Check sample rate: 16 kHz expected. Resample with librosa if needed
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        sr = 16000
        
    duration_sec = len(audio) / sr
    
    # 4. Pass numpy array directly to faster-whisper (bypassing PyAV)
    segments_gen, info = model.transcribe(
        audio,
        language="en",
        beam_size=5,
        temperature=0.0,
        condition_on_previous_text=False,
        vad_filter=False,
        word_timestamps=True,
        initial_prompt=VERBATIM_INITIAL_PROMPT
    )
    
    # Crucial: convert lazy generator to list!
    segments = list(segments_gen)
    
    transcript_parts = []
    all_words = []
    segment_logprobs = []
    segment_no_speech_probs = []
    
    for seg in segments:
        transcript_parts.append(seg.text)
        segment_logprobs.append(seg.avg_logprob)
        segment_no_speech_probs.append(seg.no_speech_prob)
        
        if seg.words:
            for w in seg.words:
                all_words.append(w)
                
    full_transcript = " ".join(transcript_parts).strip()
    num_words = len(all_words)
    speech_rate = (num_words / duration_sec) if duration_sec > 0 else 0.0
    
    # Calculate inter-word pauses (> 0.5s gaps between consecutive words)
    pauses_05 = []
    for i in range(1, len(all_words)):
        prev_end = all_words[i-1].end
        curr_start = all_words[i].start
        gap = curr_start - prev_end
        if gap > 0.5:
            pauses_05.append(gap)
            
    num_pauses_05 = len(pauses_05)
    mean_pause_sec = float(np.mean(pauses_05)) if pauses_05 else 0.0
    mean_avg_logprob = float(np.mean(segment_logprobs)) if segment_logprobs else 0.0
    mean_no_speech_prob = float(np.mean(segment_no_speech_probs)) if segment_no_speech_probs else 0.0
    
    return {
        'transcript': full_transcript,
        'duration_sec': round(duration_sec, 3),
        'num_words': num_words,
        'speech_rate': round(speech_rate, 4),
        'num_pauses_05': num_pauses_05,
        'mean_pause_sec': round(mean_pause_sec, 4),
        'mean_avg_logprob': round(mean_avg_logprob, 4),
        'mean_no_speech_prob': round(mean_no_speech_prob, 4)
    }

In [ ]:
# Batch execution with RESET_OUTPUT flag, Fail-Fast protection & Separate Failure CSV

RESET_OUTPUT = True  # Set to True to delete existing output CSVs and start fresh

transcripts_csv_path = os.path.join(OUTPUT_DIR, "transcripts.csv")
failed_csv_path = os.path.join(OUTPUT_DIR, "failed_files.csv")

# 1. Reset output files if RESET_OUTPUT is True
if RESET_OUTPUT:
    if os.path.exists(transcripts_csv_path):
        os.remove(transcripts_csv_path)
        print(f"[RESET] Removed existing output file: {transcripts_csv_path}")
    if os.path.exists(failed_csv_path):
        os.remove(failed_csv_path)
        print(f"[RESET] Removed existing failed log file: {failed_csv_path}")

# 2. Resume support: Load existing successful records if transcripts.csv exists
processed_pairs = set()
if os.path.exists(transcripts_csv_path):
    try:
        existing_df = pd.read_csv(transcripts_csv_path)
        if {'split', 'filename'}.issubset(existing_df.columns):
            for _, r in existing_df.iterrows():
                processed_pairs.add((str(r['split']), str(r['filename'])))
            print(f"[RESUME] Found existing transcripts.csv with {len(processed_pairs)} processed files.")
    except Exception as e:
        print(f"[RESUME WARNING] Could not parse existing transcripts.csv: {e}")

# 3. Build target task list
task_list = []
for _, row in train_df.iterrows():
    task_list.append(('train', row[filename_col]))
for _, row in test_df.iterrows():
    task_list.append(('test', row[filename_col]))

# Handle QUICK_TEST mode (5 train + 5 test files)
if QUICK_TEST:
    train_tasks = [t for t in task_list if t[0] == 'train'][:5]
    test_tasks = [t for t in task_list if t[0] == 'test'][:5]
    task_list = train_tasks + test_tasks
    print(f"[QUICK_TEST ACTIVE] Sliced dataset to {len(task_list)} files (5 train + 5 test).")

# Filter pending tasks
pending_tasks = [t for t in task_list if (t[0], str(t[1])) not in processed_pairs]
print(f"[BATCH] Total items: {len(task_list)} | Pending: {len(pending_tasks)}")

# Track batch progress & failures
batch_start_time = time.time()
buffer_rows = []
failed_rows = []
consecutive_failures = 0
checkpoint_freq = 25

with tqdm(total=len(pending_tasks), desc="Transcribing Audio") as pbar:
    for idx, (split_name, orig_fname) in enumerate(pending_tasks, 1):
        audio_path = build_audio_path(split_name, orig_fname)
        
        try:
            features = process_audio_file(model, audio_path)
            row_dict = {
                'split': split_name,
                'filename': orig_fname,
                **features
            }
            buffer_rows.append(row_dict)
            consecutive_failures = 0  # Reset counter on success
            
        except Exception as err:
            err_msg = str(err)
            consecutive_failures += 1
            print(f"\n[ERROR] Failed processing ({split_name}, {orig_fname}): {err_msg}")
            
            failed_rows.append({
                'split': split_name,
                'filename': orig_fname,
                'error': err_msg
            })
            
            # Fail-fast check: if the first 3 files fail in a row, halt execution immediately
            if consecutive_failures >= 3:
                # Save collected failures so far to failed_files.csv before halting
                if failed_rows:
                    pd.DataFrame(failed_rows).to_csv(failed_csv_path, index=False)
                raise RuntimeError(
                    f"FAIL-FAST HALT: First 3 audio files failed consecutively. "
                    f"Last error: {err_msg}"
                )
                
        pbar.update(1)
        
        # Checkpoint: Write successful transcriptions to transcripts.csv every 25 files
        if len(buffer_rows) >= checkpoint_freq or (idx == len(pending_tasks) and len(buffer_rows) > 0):
            new_df = pd.DataFrame(buffer_rows)
            write_header = not os.path.exists(transcripts_csv_path)
            new_df.to_csv(transcripts_csv_path, mode='a', header=write_header, index=False)
            buffer_rows = []

# Save failed files to OUTPUT_DIR/failed_files.csv if any failures occurred
if failed_rows:
    failed_df = pd.DataFrame(failed_rows)
    write_fail_header = not os.path.exists(failed_csv_path)
    failed_df.to_csv(failed_csv_path, mode='a', header=write_fail_header, index=False)

total_batch_time = time.time() - batch_start_time
num_processed = len(pending_tasks)
avg_sec = (total_batch_time / num_processed) if num_processed > 0 else 0.0

print("\n--- Transcription Batch Complete ---")
print(f"Total Time Elapsed: {total_batch_time:.2f} seconds ({total_batch_time/60:.2f} minutes)")
print(f"Processed Files:    {num_processed}")
print(f"Successful Rows:    {len(pending_tasks) - len(failed_rows)}")
print(f"Failed Files Count: {len(failed_rows)}")
if failed_rows:
    print(f"⚠️ Failed file list saved to: {failed_csv_path}")
print(f"Average Speed:      {avg_sec:.2f} seconds per audio file")

## 5. Sanity Check & Feature Analysis

In this step:
1. Print example transcripts for **Low** ($\le 2.0$), **Mid** ($2.5 – 3.5$), and **High** ($\ge 4.0$) grammar score tiers (non-overlapping groups).
2. Compute Pearson correlation between fluency features and target grammar scores.
3. Validate final output row count accounting for logged failures in `failed_files.csv` and ensure zero duplicate `(split, filename)` records.

> **Known Edge Case Note (`train/audio_709.wav`)**: During full transcription, 1 file (`train/audio_709.wav`) triggered an internal NumPy boolean indexing bug in faster-whisper's word timestamp alignment algorithm (`boolean index did not match indexed array along axis 0`). It was automatically logged in `failed_files.csv` and excluded from `transcripts.csv` without crashing the batch.

In [ ]:
# Load full generated transcripts CSV
transcripts_df = pd.read_csv(transcripts_csv_path)
print(f"Loaded transcripts.csv with shape: {transcripts_df.shape}\n")

# Merge train labels into transcripts dataframe for sanity checks
train_labels_df = train_df[[filename_col, label_col]].copy()
train_labels_df.rename(columns={filename_col: 'filename', label_col: 'target_score'}, inplace=True)

merged_train = pd.merge(
    transcripts_df[transcripts_df['split'] == 'train'],
    train_labels_df,
    on='filename',
    how='inner'
)

print(f"Merged training rows available for sanity check: {len(merged_train)}")

# Sample transcripts across non-overlapping target tiers: Low (<=2.0), Mid (2.5-3.5), High (>=4.0)
print("\n=======================================================")
print("  QUALITATIVE SAMPLE TRANSCRIPTS BY GRAMMAR SCORE TIER")
print("=======================================================")

def print_samples(df_tier, tier_name, count=3):
    print(f"\n--- Tier: {tier_name} (Sample Count: {len(df_tier)}) ---")
    samples = df_tier.head(count)
    for idx, r in samples.iterrows():
        print(f"\n[File: {r['filename']} | Score: {r['target_score']}]")
        print(f"Transcript: \"{r['transcript']}\"")
        print(f"Stats: words={r['num_words']}, rate={r['speech_rate']} w/s, pauses_0.5s={r['num_pauses_05']}")

low_tier = merged_train[merged_train['target_score'] <= 2.0]
mid_tier = merged_train[(merged_train['target_score'] >= 2.5) & (merged_train['target_score'] <= 3.5)]
high_tier = merged_train[merged_train['target_score'] >= 4.0]

print_samples(low_tier, "Low Score (<= 2.0)")
print_samples(mid_tier, "Mid Score (2.5 - 3.5)")
print_samples(high_tier, "High Score (>= 4.0)")

In [ ]:
# Compute Pearson Correlation between target grammar score and audio/fluency features
feature_cols = ['num_words', 'speech_rate', 'num_pauses_05', 'mean_pause_sec', 'mean_avg_logprob', 'mean_no_speech_prob']

if len(merged_train) > 1:
    corr_matrix = merged_train[feature_cols + ['target_score']].corr(method='pearson')
    target_corr = corr_matrix['target_score'].drop('target_score').sort_values(ascending=False)
    
    print("\n--- Pearson Correlation with Target Grammar Score ---")
    corr_table = pd.DataFrame({'Pearson_r': target_corr})
    display(corr_table)
    
    # Plot Scatter Plot Grid
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()
    
    for i, col in enumerate(feature_cols):
        sns.regplot(data=merged_train, x=col, y='target_score', ax=axes[i], scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
        axes[i].set_title(f"{col} vs Score (r = {target_corr[col]:.3f})")
        axes[i].grid(True, alpha=0.3)
        
    plt.tight_layout()
    plt.show()
else:
    print("\n[NOTE] Not enough merged training samples present for correlation scatter plots in this run.")

In [ ]:
# Output Validation & Integrity Assertions
print("--- Final Dataset Integrity Assertions ---")

# 1. Assert no duplicate (split, filename) pairs
duplicates = transcripts_df.duplicated(subset=['split', 'filename']).sum()
print(f"Duplicate (split, filename) pairs: {duplicates}")
assert duplicates == 0, f"Error: Found {duplicates} duplicate (split, filename) entries!"

# 2. Inspect failed files log if present
failed_csv_path = os.path.join(OUTPUT_DIR, "failed_files.csv")
failed_train_count = 0
failed_test_count = 0
if os.path.exists(failed_csv_path):
    try:
        failed_df = pd.read_csv(failed_csv_path)
        failed_train_count = (failed_df['split'] == 'train').sum()
        failed_test_count = (failed_df['split'] == 'test').sum()
        print(f"\n[FAILED FILES LOG] Found {len(failed_df)} failed file(s) logged in failed_files.csv:")
        for _, r in failed_df.iterrows():
            print(f"  - Split: {r['split']} | Filename: {r['filename']} | Error: {r.get('error', 'N/A')}")
    except Exception as e:
        print(f"[FAILED FILES WARNING] Could not parse failed_files.csv: {e}")

# 3. Row count assertions (conditional on QUICK_TEST flag)
train_count = (transcripts_df['split'] == 'train').sum()
test_count = (transcripts_df['split'] == 'test').sum()

expected_train = 769 - failed_train_count
expected_test = 216 - failed_test_count

print(f"\nCurrent Row Counts -> Train: {train_count} (Expected: {expected_train}) | Test: {test_count} (Expected: {expected_test}) | Total: {len(transcripts_df)}")

if not QUICK_TEST:
    # Full run strict assertions accounting for logged failures
    print("[FULL RUN VERIFICATION] Checking exact dataset row counts accounting for failed files...")
    assert train_count == expected_train, f"Assertion Failed! Expected {expected_train} train rows, got {train_count}"
    assert test_count == expected_test, f"Assertion Failed! Expected {expected_test} test rows, got {test_count}"
    print(f"✅ Full Run Integrity Check PASSED! Exactly {train_count} train + {test_count} test rows validated.")
else:
    # Quick test notification
    print("ℹ️ QUICK_TEST=True: Skipping 769/216 row assertion (Quick test processed subset).")
    assert train_count > 0 and test_count > 0, "Quick test failed to process train or test rows!"
    print("✅ Quick Test Integrity Check PASSED!")

## 6. Summary & Downstream Pipeline Roadmap

### What `01_transcribe.ipynb` Produced:
- Output CSV saved to `/kaggle/working/transcripts.csv`.
- Failed files log (if any) saved to `/kaggle/working/failed_files.csv`.
- Each successful record contains:
  - `split`: 'train' or 'test'.
  - `filename`: exact filename string as present in the competition CSV.
  - `transcript`: verbatim text output generated by Whisper `large-v3`.
  - `duration_sec`: audio duration in seconds.
  - `num_words`: total word count.
  - `speech_rate`: words per second.
  - `num_pauses_05`: number of speech pauses > 0.5 seconds.
  - `mean_pause_sec`: average pause duration.
  - `mean_avg_logprob`: Whisper model confidence metric.
  - `mean_no_speech_prob`: Whisper silence/non-speech probability.

---

### How `02_model.ipynb` Will Use `transcripts.csv`:
1. **Text Embedding & Fine-Tuning**: Pass verbatim transcripts into pretrained DeBERTa-v3 / RoBERTa models to extract deep linguistic features and predict grammar scores directly from text.
2. **Feature Fusion**: Combine text model predictions/embeddings with extracted audio fluency metrics (`speech_rate`, `num_pauses_05`, `mean_pause_sec`).
3. **Cross-Validation**: Train GBDT models (LightGBM/XGBoost/CatBoost) using Stratified K-Fold CV.
4. **Evaluation**: Optimize and report **Pearson Correlation** and **RMSE** metrics.
5. **Submission Generation**: Output final test predictions matching the format in `sample_submission.csv`.